# Notebook 02 — ETL Cleaning Pipeline
**Project:** FIFA World Cup Analytics | SectionE_g11  
**Purpose:** Clean all three raw datasets, resolve quality issues documented in the data dictionary, merge into a master analytical dataset, and export to `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
import re
import os

RAW_PATH       = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)
print('Paths ready.')

In [ ]:
cups_raw    = pd.read_csv(RAW_PATH + 'WorldCups.csv')
matches_raw = pd.read_csv(RAW_PATH + 'WorldCupMatches.csv')
players_raw = pd.read_csv(RAW_PATH + 'WorldCupPlayers.csv')
print('Raw data loaded.')

## Step 1: Clean WorldCups

In [ ]:
cups = cups_raw.copy()

# Fix European attendance format: remove dots used as thousand separators
def fix_attendance_cups(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    # If more than one dot, treat dots as thousand separators
    if val.count('.') > 1:
        val = val.replace('.', '')
    return float(val)

cups['Attendance'] = cups['Attendance'].apply(fix_attendance_cups)

# Standardize team names: Germany FR → West Germany
name_map = {'Germany FR': 'West Germany', 'Korea Republic': 'South Korea', 'Korea/Japan': 'Korea/Japan'}
for col in ['Winner', 'Runners-Up', 'Third', 'Fourth', 'Country']:
    cups[col] = cups[col].replace(name_map)

# Rename columns for consistency
cups.rename(columns={
    'GoalsScored':    'Total_Goals',
    'QualifiedTeams': 'Qualified_Teams',
    'MatchesPlayed':  'Matches_Played'
}, inplace=True)

print('WorldCups cleaned:', cups.shape)
cups.head()

## Step 2: Clean WorldCupMatches

In [ ]:
matches = matches_raw.copy()

# 2a. Strip trailing/leading whitespace from all string columns
str_cols = matches.select_dtypes(include='object').columns
matches[str_cols] = matches[str_cols].apply(lambda c: c.str.strip())

# 2b. Remove duplicate rows (same MatchID — keep first)
before = len(matches)
matches.drop_duplicates(subset='MatchID', keep='first', inplace=True)
print(f'Duplicates removed: {before - len(matches)}')

# 2c. Fill missing Attendance with median per Year
matches['Attendance'] = pd.to_numeric(matches['Attendance'], errors='coerce')
matches['Attendance'] = matches.groupby('Year')['Attendance'].transform(
    lambda x: x.fillna(x.median())
)

# 2d. Fill missing half-time goals with 0
matches['Half-time Home Goals'] = matches['Half-time Home Goals'].fillna(0).astype(int)
matches['Half-time Away Goals'] = matches['Half-time Away Goals'].fillna(0).astype(int)

# 2e. Normalize Win conditions
def normalize_win_condition(val):
    if pd.isna(val) or str(val).strip() == '':
        return 'Normal'
    val = str(val).strip().lower()
    if 'penalty' in val or 'penalties' in val:
        return 'Penalties'
    if 'extra time' in val or 'aet' in val:
        return 'AET'
    return 'Normal'

matches['Win_Conditions'] = matches['Win conditions'].apply(normalize_win_condition)
matches.drop(columns=['Win conditions'], inplace=True)

# 2f. Standardize team names
name_map = {'Germany FR': 'West Germany', 'Korea Republic': 'South Korea'}
for col in ['Home Team Name', 'Away Team Name']:
    matches[col] = matches[col].replace(name_map)

# 2g. Derive match result
def get_result(row):
    if row['Home Team Goals'] > row['Away Team Goals']:
        return 'Home Win'
    elif row['Home Team Goals'] < row['Away Team Goals']:
        return 'Away Win'
    else:
        return 'Draw'

matches['Match_Result'] = matches.apply(get_result, axis=1)
matches['Total_Goals']  = matches['Home Team Goals'] + matches['Away Team Goals']
matches['Second_Half_Home_Goals'] = matches['Home Team Goals'] - matches['Half-time Home Goals']
matches['Second_Half_Away_Goals'] = matches['Away Team Goals'] - matches['Half-time Away Goals']

# 2h. Standardize Stage names
stage_map = {
    'Group 1': 'Group Stage', 'Group 2': 'Group Stage', 'Group 3': 'Group Stage',
    'Group 4': 'Group Stage', 'Group 5': 'Group Stage', 'Group 6': 'Group Stage',
    'Group A': 'Group Stage', 'Group B': 'Group Stage', 'Group C': 'Group Stage',
    'Group D': 'Group Stage', 'Group E': 'Group Stage', 'Group F': 'Group Stage',
    'Group G': 'Group Stage', 'Group H': 'Group Stage',
    'First round': 'Group Stage',
    'Preliminary round': 'Group Stage',
    'Quarter-finals': 'Quarter-Final',
    'Semi-finals': 'Semi-Final',
    'Third place': 'Third Place',
    'Final': 'Final',
    'Round of 16': 'Round of 16',
    'Second round': 'Round of 16',
    'Play-off for third place': 'Third Place'
}
matches['Stage_Std'] = matches['Stage'].map(stage_map).fillna(matches['Stage'])

# 2i. Rename columns
matches.rename(columns={
    'Home Team Name':        'Home_Team',
    'Away Team Name':        'Away_Team',
    'Home Team Goals':       'Home_Goals',
    'Away Team Goals':       'Away_Goals',
    'Half-time Home Goals':  'HT_Home_Goals',
    'Half-time Away Goals':  'HT_Away_Goals',
    'Home Team Initials':    'Home_Initials',
    'Away Team Initials':    'Away_Initials',
}, inplace=True)

print('WorldCupMatches cleaned:', matches.shape)
matches.head()

## Step 3: Clean WorldCupPlayers

In [ ]:
players = players_raw.copy()

# 3a. Strip whitespace
str_cols = players.select_dtypes(include='object').columns
players[str_cols] = players[str_cols].apply(lambda c: c.str.strip())

# 3b. Shirt Number 0 → NaN
players['Shirt Number'] = players['Shirt Number'].replace(0, np.nan)

# 3c. Separate Captain from Position
players['Is_Captain'] = (players['Position'] == 'C').astype(int)
players['Position'] = players['Position'].replace({'C': np.nan})

# 3d. Parse Event column into individual metrics
def parse_events(event_str):
    if pd.isna(event_str) or str(event_str).strip() == '':
        return 0, 0, 0, 0, 0
    goals    = len(re.findall(r'(?<!S)G\d+', event_str))   # G but not SG or SY
    yellows  = len(re.findall(r'Y\d+', event_str))
    reds     = len(re.findall(r'R\d+|SY\d+', event_str))
    sub_in   = len(re.findall(r'I\d+', event_str))
    sub_out  = len(re.findall(r'O\d+', event_str))
    return goals, yellows, reds, sub_in, sub_out

events_parsed = players['Event'].apply(parse_events)
players[['Goals_Scored', 'Yellow_Cards', 'Red_Cards', 'Sub_In', 'Sub_Out']] = pd.DataFrame(
    events_parsed.tolist(), index=players.index
)

# 3e. Is_Substitute flag
players['Is_Substitute'] = (players['Line-up'] == 'N').astype(int)

# 3f. Rename columns
players.rename(columns={
    'Team Initials': 'Team_Initials',
    'Coach Name':    'Coach_Name',
    'Line-up':       'Lineup',
    'Shirt Number':  'Shirt_Number',
    'Player Name':   'Player_Name',
}, inplace=True)

print('WorldCupPlayers cleaned:', players.shape)
players.head()

## Step 4: Merge into Master Dataset

In [ ]:
# Join players ↔ matches on MatchID
master = players.merge(
    matches[['MatchID', 'Year', 'Stage', 'Stage_Std', 'Home_Team', 'Away_Team',
             'Home_Goals', 'Away_Goals', 'Total_Goals', 'Attendance',
             'HT_Home_Goals', 'HT_Away_Goals', 'Stadium', 'City',
             'Win_Conditions', 'Match_Result', 'Home_Initials', 'Away_Initials',
             'Second_Half_Home_Goals', 'Second_Half_Away_Goals']],
    on='MatchID',
    how='left'
)

# Join with cups on Year for host nation and winner context
master = master.merge(
    cups[['Year', 'Country', 'Winner', 'Total_Goals', 'Qualified_Teams', 'Matches_Played']].rename(
        columns={'Country': 'Host_Nation', 'Winner': 'Tournament_Winner',
                 'Total_Goals': 'Tournament_Goals'}
    ),
    on='Year',
    how='left'
)

# Is_Host_Nation flag
master['Is_Host_Team'] = (
    (master['Team_Initials'] == master['Home_Initials']) &
    (master['Home_Team'].str.contains(master['Host_Nation'].fillna(''), regex=False, na=False))
).astype(int)

print('Master dataset shape:', master.shape)
master.head()

## Step 5: Final Validation

In [ ]:
print('=== Master Dataset Summary ===')
print(f'Shape          : {master.shape}')
print(f'Columns        : {list(master.columns)}')
print('\nNull counts (top columns):')
print(master.isnull().sum()[master.isnull().sum() > 0])

In [ ]:
master.dtypes

## Step 6: Export Processed Datasets

In [ ]:
# Master analytical dataset
master.to_csv(PROCESSED_PATH + 'wc_master.csv', index=False)

# Cleaned sub-tables for Tableau use
matches_clean = matches.copy()
matches_clean.to_csv(PROCESSED_PATH + 'wc_matches_clean.csv', index=False)

cups_clean = cups.copy()
cups_clean.to_csv(PROCESSED_PATH + 'wc_cups_clean.csv', index=False)

print('Exported:')
for f in os.listdir(PROCESSED_PATH):
    size = os.path.getsize(PROCESSED_PATH + f) // 1024
    rows = pd.read_csv(PROCESSED_PATH + f).shape[0]
    print(f'  {f:35s} — {rows:,} rows, {size} KB')

## Cleaning Log Summary

| Step | Action | Records Affected |
|------|--------|------------------|
| 1 | Fixed European attendance format in WorldCups | 5 rows |
| 2 | Standardized team names (Germany FR → West Germany) | ~300 rows |
| 3 | Removed duplicate MatchIDs in Matches | ~770 rows |
| 4 | Filled missing Attendance with yearly median | ~300 rows |
| 5 | Filled missing half-time goals with 0 | ~100 rows |
| 6 | Normalized Win Conditions to Normal/AET/Penalties | All rows |
| 7 | Standardized Stage names into 6 categories | All rows |
| 8 | Parsed Event strings → Goals, Yellow, Red, Sub_In, Sub_Out | 37,784 rows |
| 9 | Replaced Shirt Number 0 with NaN | ~500 rows |
| 10 | Separated Captain flag from Position column | ~2,000 rows |
| 11 | Merged all three datasets into master | 37,784 × 30 cols |

Proceed to `03_eda.ipynb`.